In [1]:
%load_ext autoreload
%autoreload 2
import sys, os
sys.path.insert(0, os.path.join('.', 'code'))   
from utils import BatchIndex,get_mgrid,fast_random_choice,count_params,cleanup,seed_everything,adjust_lr
from dataio import ScalarDataSet
from inr import CoordNet,trainNet,inf

In [3]:
import argparse

p = argparse.ArgumentParser()
p.add_argument('--no-cuda', action='store_true', default=False,
                    help='disables CUDA training')
p.add_argument('--gpu', type=str,default='0')
p.add_argument('--seed', type=int, default=42)
p.add_argument('--fp16', action="store_true")
p.add_argument('--compile', action="store_true")
# General training options
p.add_argument('--downsample_factor', type=int, default=4, metavar='N',
                    help='downsample factor')
p.add_argument('--batch_size', type=int, default=8000)
p.add_argument('--lr', type=float, default=1e-5, help='learning rate. default=1e-4')
p.add_argument('--num_epochs', type=int, default=200,
               help='Number of epochs to train for.')
p.add_argument('--checkpoint', type=int, default=50,
               help='checkpoint is saved.')
p.add_argument('--ckpt', type=str,default="/INR/checkpoints/-0--200.pth",
               help='checkpoint path.')
p.add_argument('--dataset', type=str, default='fivejets',
               help='Scalar dataset; one of (Vortex, combustion)')
p.add_argument('--result_dir', type=str, default='./', metavar='N',
                    help='the path where we stored the synthesized data')
p.add_argument('--interval', type=int, default=0, metavar='N',
                    help='temporal upscaling factor')
p.add_argument('--active', type=str, default='sine', metavar='N',
                    help='active function')
p.add_argument('--lr_s', type=str, default='cosine', help='step or exp')
p.add_argument('--mode', type=str, default='inf', metavar='N',
                    help='the path where we stored the synthesized data')
opt = p.parse_known_args()[0]

import torch
import os
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
os.environ['CUDA_VISIBLE_DEVICES'] = opt.gpu
os.environ["KMP_DUPLICATE_LIB_OK"]  =  "TRUE"

opt.cuda = not opt.no_cuda and torch.cuda.is_available()
seed_everything(opt.seed)

torch.set_float32_matmul_precision('high')

def main():
    print('FP16 enbled: ', opt.fp16)
    print('Compile enbled: ', opt.compile)
    Data = ScalarDataSet(opt)
    Model = CoordNet(4,1,init_features=64, num_res=7)
    if opt.mode in ['inf', 'ue']:
        ckpt = './'+opt.dataset+opt.ckpt
        Model.load_state_dict(torch.load(ckpt))
    if opt.compile:
        Model.compile()
    Model.cuda()

    if opt.mode == 'train':
        print('Initalize Model Successfully using Sine Function!')
        trainNet(Model,opt,Data)
    elif opt.mode == 'inf':
        inf(Model, Data,opt)
    
if __name__== "__main__":
    main()


FP16 enbled:  False
Compile enbled:  False


100%|██████████| 263/263 [00:00<00:00, 562.33it/s]


In [4]:
import numpy as np
import torch
data_name = 'fivejets'
origin_dir = './dataset/' + data_name + '/'
recons_dir = './' + data_name + '/INR/outputs/inference/'
psnr = 0
k=0
psnr_fn_paper = lambda gt, pred, diff: 10. * torch.log10(diff**2 / torch.mean((gt-pred)**2))
for i in range(1,2):
    gt = np.fromfile(origin_dir + '{:04d}.raw'.format(i),dtype=np.float32)
    d = np.fromfile(recons_dir + "{:04d}.dat".format(i),dtype=np.float32)
    gt = 2*(gt-np.min(gt))/(np.max(gt)-np.min(gt))-1
    d = torch.from_numpy(d)
    gt = torch.from_numpy(gt)    
    diff = gt.max() - gt.min()
    
    psnr_volume = psnr_fn_paper(gt, d, diff)
    print(str(i)+":"+str(psnr_volume.item()))
    psnr+=psnr_volume.item()
    k+=1
# print(psnr/k)

1:43.05831527709961
